# Forklift Movie Preprocess

元動画から front / rear 動画を切り出し、
同じトリム区間の音声 wav も `data/movie_preprocess/<category>/<environment>/` に保存します。
`data/frame_cut.txt` に指定がある動画は、最初に前後フレームをカットしてから無地フレームをトリムします。


このノートブックは、元動画を学習・推論で扱いやすい `front.mp4` / `rear.mp4` / `audio.wav` にそろえる前処理です。黒帯除去、前後カメラ分割、trim、音声抽出、manifest 保存までを同じルールで行います。


## 0. パッケージ


In [ ]:
# ------------------------------------------------------------
# セル概要: パッケージ確認
# ------------------------------------------------------------
# - 動画前処理に必要な OpenCV / numpy / pandas / matplotlib などを確認します。
# - ffmpeg / ffprobe はPythonパッケージではなく実行ファイルなので、後続セルで存在確認します。
# ------------------------------------------------------------

# %pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 pandas==2.2.3 opencv-python-headless==4.11.0.86


In [ ]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - 動画読み書き、音声抽出、manifest 作成、ファイル削除に使う標準ライブラリと OpenCV を読み込みます。
# - 以降のセルでは前処理済み動画と音声を同じルールで出力します。
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
from typing import Any

import shutil
import subprocess

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
# ------------------------------------------------------------
# セル概要: 前処理設定
# ------------------------------------------------------------
# - 入力動画の場所、出力先、FPS、重複フレーム除去、黒帯除去、音声サンプリング周波数をまとめます。
# - ここを変えると全動画の出力仕様が変わるため、manifest にも設定値を記録します。
# ------------------------------------------------------------

DATA_DIR = Path("../data")
ORI_DATA_DIR = DATA_DIR / "ori"
INPUT_VIDEO_DIRS = {
    "normal": ORI_DATA_DIR / "normal",
    "anomaly": ORI_DATA_DIR / "anomaly",
}
OUTDOOR_LIST_PATH = DATA_DIR / "outdoor_list.txt"
FRAME_CUT_PATH = DATA_DIR / "frame_cut.txt"
MOVIE_PREPROCESS_OUTPUT_DIR = DATA_DIR / "movie_preprocess"
MOVIE_PREPROCESS_MANIFEST_PATH = MOVIE_PREPROCESS_OUTPUT_DIR / "movie_preprocess_manifest.csv"

VIDEO_PROCESSING_ENABLED = True
MOVIE_PREPROCESS_OVERWRITE = False
MOVIE_PREPROCESS_DELETE_STALE = True
MOVIE_PREPROCESS_FOURCC = "mp4v"
MOVIE_PREPROCESS_DEDUP_FRAMES = False
MOVIE_PREPROCESS_DEDUP_DIFF_THRESHOLD = 0.003
MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH = 160
MOVIE_PREPROCESS_DEDUP_MAX_SKIPPED_RUN = None
MOVIE_PREPROCESS_MIN_OUTPUT_FPS = 1.0
VIDEO_SAMPLE_FPS = 2
VIDEO_MAX_SAMPLE_FRAMES = 200
VIDEO_LETTERBOX_BLACK_THRESHOLD = 8
VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD = 0.95
VIDEO_UNIFORM_COLOR_TOLERANCE = 30.0
AUDIO_SAMPLE_RATE = 22050
AUDIO_MONO_CHANNELS = 1

FFMPEG_PATH = shutil.which("ffmpeg")
FFPROBE_PATH = shutil.which("ffprobe")
if FFMPEG_PATH is None:
    raise FileNotFoundError("ffmpeg is required but was not found in PATH.")
if FFPROBE_PATH is None:
    raise FileNotFoundError("ffprobe is required but was not found in PATH.")

## 1. ユーティリティ


In [ ]:
# ------------------------------------------------------------
# セル概要: 前処理ユーティリティ
# ------------------------------------------------------------
# - 動画メタデータ取得、黒帯検出、front/rear 分割、重複フレーム除去、古い成果物掃除などの補助関数群です。
# - 単体関数を小さく分け、バッチ処理本体では「何をするか」が読めるようにしています。
# ------------------------------------------------------------

# 関数メモ: ensure_parent_directory
# - 出力ファイルを書き込む前に親ディレクトリを作成します。
# - 関数本体のあちこちで `mkdir` を重複させないための小さな共通処理です。

def ensure_parent_directory(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


# 関数メモ: read_video_metadata
# - OpenCV で動画の幅・高さ・FPS・フレーム数・長さを取得します。
# - 前処理の可否判定、manifest 記録、trim 範囲の計算に使います。

def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "video_path": str(video_path),
        "frame_count": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    metadata["duration_sec"] = (
        metadata["frame_count"] / metadata["fps"] if metadata["fps"] > 0 else np.nan
    )
    capture.release()
    return metadata


# 関数メモ: iter_sampled_frames
# - 動画全体を毎フレーム処理せず、指定FPS相当でサンプリングしてフレームを返します。
# - 黒帯や単色フレームの傾向を低コストで推定するための読み出し関数です。

def iter_sampled_frames(
    video_path: Path,
    sampling_fps: int = VIDEO_SAMPLE_FPS,
    max_frames: int | None = VIDEO_MAX_SAMPLE_FRAMES,
    start_frame_index: int = 0,
    end_frame_index: int | None = None,
) -> list[np.ndarray]:
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 1.0
    step = max(int(round(native_fps / max(sampling_fps, 1))), 1)
    end_frame_index = metadata["frame_count"] if end_frame_index is None else end_frame_index

    frames: list[np.ndarray] = []
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    while capture.isOpened():
        success, frame_bgr = capture.read()
        if not success:
            break
        if frame_index >= end_frame_index:
            break
        if frame_index < start_frame_index:
            frame_index += 1
            continue
        if (frame_index - start_frame_index) % step == 0:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames) >= max_frames:
                break
        frame_index += 1

    capture.release()
    return frames


# 関数メモ: detect_side_letterbox_bounds
# - 左右に入った黒帯・単色帯の境界を、列ごとの暗さや色の均一さから推定します。
# - 上下2画面結合動画を分割する前に、不要な左右余白を落とすために使います。

def detect_side_letterbox_bounds(
    stacked_frame: np.ndarray,
    black_threshold: int = VIDEO_LETTERBOX_BLACK_THRESHOLD,
    min_non_black_ratio: float = 0.01,
) -> tuple[int, int, dict[str, int]]:
    gray = cv2.cvtColor(stacked_frame, cv2.COLOR_RGB2GRAY)
    non_black_ratio = (gray > black_threshold).mean(axis=0)
    valid_columns = np.where(non_black_ratio > min_non_black_ratio)[0]

    if valid_columns.size == 0:
        crop_left = 0
        crop_right = stacked_frame.shape[1]
    else:
        crop_left = int(valid_columns[0])
        crop_right = int(valid_columns[-1]) + 1

    if crop_right <= crop_left:
        crop_left = 0
        crop_right = stacked_frame.shape[1]

    return crop_left, crop_right, {
        "original_width": int(stacked_frame.shape[1]),
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(stacked_frame.shape[1] - crop_right),
    }


# 関数メモ: apply_side_crop
# - 推定した左端・右端の crop 位置でフレームを切り出します。
# - crop 範囲が無効な場合は元フレームを返し、異常な設定で処理が壊れにくいようにします。

def apply_side_crop(stacked_frame: np.ndarray, crop_left: int, crop_right: int) -> np.ndarray:
    return stacked_frame[:, crop_left:crop_right, :]


# 関数メモ: choose_common_side_letterbox_crop
# - 複数サンプルフレームから共通して安全に使える左右 crop 幅を決めます。
# - フレームごとに crop が揺れると出力動画が不安定になるため、動画単位の共通値にします。

def choose_common_side_letterbox_crop(stacked_frames: list[np.ndarray]) -> dict[str, int]:
    if not stacked_frames:
        return {
            "frame_count": 0,
            "original_width": 0,
            "crop_left": 0,
            "crop_right": 0,
            "cropped_width": 0,
            "removed_left_px": 0,
            "removed_right_px": 0,
        }

    crop_lefts: list[int] = []
    crop_rights: list[int] = []
    original_width = int(stacked_frames[0].shape[1])
    for stacked_frame in stacked_frames:
        crop_left, crop_right, _ = detect_side_letterbox_bounds(stacked_frame)
        crop_lefts.append(crop_left)
        crop_rights.append(crop_right)

    crop_left = int(np.median(crop_lefts))
    crop_right = int(np.median(crop_rights))
    crop_left = max(0, min(crop_left, original_width - 1))
    crop_right = max(crop_left + 1, min(crop_right, original_width))

    return {
        "frame_count": int(len(stacked_frames)),
        "original_width": original_width,
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(original_width - crop_right),
    }


# 関数メモ: preprocess_stacked_frames_remove_letterbox
# - サンプルフレーム群から左右黒帯を推定し、crop 情報をまとめます。
# - 以降の実フレーム処理ではこの crop 情報を使って一貫した切り出しを行います。

def preprocess_stacked_frames_remove_letterbox(
    stacked_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], dict[str, int]]:
    if not stacked_frames:
        return [], choose_common_side_letterbox_crop(stacked_frames)

    crop_info = choose_common_side_letterbox_crop(stacked_frames)
    cropped_frames = [
        apply_side_crop(stacked_frame, crop_info["crop_left"], crop_info["crop_right"])
        for stacked_frame in stacked_frames
    ]
    return cropped_frames, crop_info


# 関数メモ: split_front_rear_frame
# - 上下に積まれた1枚のフレームを front と rear の2枚に分割します。
# - 前処理済み出力を `*_front.mp4` / `*_rear.mp4` に分ける中心処理です。

def split_front_rear_frame(stacked_frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height = stacked_frame.shape[0]
    midpoint = height // 2
    return stacked_frame[:midpoint, :, :], stacked_frame[midpoint:, :, :]


# 関数メモ: is_nearly_uniform_frame
# - 黒画面や単色に近いフレームかどうかを判定します。
# - 冒頭・末尾の無効フレーム trim や、画面がほぼ情報を持たない区間の除外に使います。

def is_nearly_uniform_frame(
    frame: np.ndarray,
    dominant_ratio_threshold: float = VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD,
    color_tolerance: float = VIDEO_UNIFORM_COLOR_TOLERANCE,
) -> bool:
    pixels = frame.reshape(-1, 3).astype(np.float32)
    representative_color = np.median(pixels, axis=0)
    pixel_deviation = np.abs(pixels - representative_color).max(axis=1)
    near_representative_ratio = float((pixel_deviation <= color_tolerance).mean())
    return near_representative_ratio >= dominant_ratio_threshold


# 関数メモ: estimate_full_video_uniform_trim
# - 動画全体をサンプリングし、前後の単色に近い区間を trim する範囲を推定します。
# - 録画開始・終了時の黒画面や固定色画面を学習・推論対象から外すための処理です。

def estimate_full_video_uniform_trim(
    video_path: Path,
    crop_info: dict[str, int],
    start_frame_index: int = 0,
    end_frame_index: int | None = None,
) -> dict[str, int]:
    metadata = read_video_metadata(video_path)
    end_frame_index = metadata["frame_count"] if end_frame_index is None else end_frame_index
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    total_frame_pairs = 0
    front_uniform_start = 0
    rear_uniform_start = 0
    front_uniform_end = 0
    rear_uniform_end = 0
    front_start_open = True
    rear_start_open = True

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break
            current_frame_index = int(capture.get(cv2.CAP_PROP_POS_FRAMES)) - 1
            if current_frame_index >= end_frame_index:
                break
            if current_frame_index < start_frame_index:
                continue

            cropped_frame_bgr = apply_side_crop(
                stacked_frame_bgr,
                crop_info["crop_left"],
                crop_info["crop_right"],
            )
            front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)
            front_is_uniform = is_nearly_uniform_frame(front_frame_bgr)
            rear_is_uniform = is_nearly_uniform_frame(rear_frame_bgr)

            if front_start_open and front_is_uniform:
                front_uniform_start += 1
            else:
                front_start_open = False

            if rear_start_open and rear_is_uniform:
                rear_uniform_start += 1
            else:
                rear_start_open = False

            front_uniform_end = front_uniform_end + 1 if front_is_uniform else 0
            rear_uniform_end = rear_uniform_end + 1 if rear_is_uniform else 0
            total_frame_pairs += 1
    finally:
        capture.release()

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)
    remaining_frame_pairs = max(total_frame_pairs - removed_from_start - removed_from_end, 0)

    return {
        "input_frame_pairs": int(total_frame_pairs),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(remaining_frame_pairs),
    }


# 関数メモ: create_mp4_writer
# - OpenCV の VideoWriter を作成し、指定サイズ・FPSのMP4を書ける状態にします。
# - writer が開けない場合は即座に例外にして、空の出力が残る問題を避けます。

def create_mp4_writer(output_path: Path, fps: float, frame_shape: tuple[int, ...]) -> cv2.VideoWriter:
    ensure_parent_directory(output_path)
    height, width = frame_shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*MOVIE_PREPROCESS_FOURCC)
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (int(width), int(height)))
    if not writer.isOpened():
        raise RuntimeError(f"Failed to open video writer: {output_path}")
    return writer


# 関数メモ: prepare_frame_for_dedup
# - 重複フレーム判定用に、フレームを小さなグレースケール画像へ変換します。
# - 見た目がほぼ同じかを高速に比較するため、解析用の軽量表現へ落とします。

def prepare_frame_for_dedup(
    frame_bgr: np.ndarray,
    resize_width: int | None = MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH,
) -> np.ndarray:
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    if resize_width is not None and resize_width > 0 and gray.shape[1] > resize_width:
        scale = resize_width / gray.shape[1]
        resize_height = max(1, int(round(gray.shape[0] * scale)))
        gray = cv2.resize(gray, (int(resize_width), resize_height), interpolation=cv2.INTER_AREA)
    return gray


# 関数メモ: compute_dedup_frame_diff_score
# - 連続フレーム間の差分量を0〜1程度のスコアとして計算します。
# - この値が小さいフレームを重複候補として扱い、必要なら間引きます。

def compute_dedup_frame_diff_score(current_gray: np.ndarray, previous_gray: np.ndarray) -> float:
    if current_gray.shape != previous_gray.shape:
        current_gray = cv2.resize(
            current_gray,
            (previous_gray.shape[1], previous_gray.shape[0]),
            interpolation=cv2.INTER_AREA,
        )
    return float(np.mean(cv2.absdiff(current_gray, previous_gray)) / 255.0)


# 関数メモ: select_nonduplicate_frame_indices
# - 重複に近いフレームを除外し、残すフレーム番号を選びます。
# - 長い停止区間で同じ画面が大量に入る場合の出力サイズと処理量を抑えます。

def select_nonduplicate_frame_indices(
    video_path: Path,
    start_frame_index: int,
    end_frame_index: int,
    crop_info: dict[str, int],
    native_fps: float,
    enabled: bool = MOVIE_PREPROCESS_DEDUP_FRAMES,
    diff_threshold: float = MOVIE_PREPROCESS_DEDUP_DIFF_THRESHOLD,
    resize_width: int | None = MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH,
    max_skipped_run: int | None = MOVIE_PREPROCESS_DEDUP_MAX_SKIPPED_RUN,
) -> tuple[list[int], dict[str, Any]]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    kept_indices: list[int] = []
    previous_source_gray: np.ndarray | None = None
    duplicate_frame_pairs = 0
    skipped_run = 0
    diff_scores: list[float] = []
    frame_index = 0

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break
            if frame_index >= end_frame_index:
                break
            if frame_index < start_frame_index:
                frame_index += 1
                continue

            cropped_frame_bgr = apply_side_crop(
                stacked_frame_bgr,
                crop_info["crop_left"],
                crop_info["crop_right"],
            )
            current_gray = prepare_frame_for_dedup(cropped_frame_bgr, resize_width=resize_width)

            force_keep = max_skipped_run is not None and skipped_run >= max_skipped_run
            if not enabled or previous_source_gray is None:
                keep_frame = True
                diff_score = np.nan
            else:
                diff_score = compute_dedup_frame_diff_score(current_gray, previous_source_gray)
                diff_scores.append(diff_score)
                keep_frame = diff_score > diff_threshold or force_keep

            if keep_frame:
                kept_indices.append(frame_index)
                skipped_run = 0
            else:
                duplicate_frame_pairs += 1
                skipped_run += 1
            previous_source_gray = current_gray

            frame_index += 1
    finally:
        capture.release()

    input_frame_pairs = max(end_frame_index - start_frame_index, 0)
    source_duration_sec = input_frame_pairs / native_fps if native_fps > 0 else 0.0
    output_fps = native_fps
    if enabled and kept_indices and source_duration_sec > 0:
        output_fps = len(kept_indices) / source_duration_sec
        output_fps = max(float(output_fps), float(MOVIE_PREPROCESS_MIN_OUTPUT_FPS))

    finite_diff_scores = np.asarray(diff_scores, dtype=float)
    info = {
        "dedup_enabled": bool(enabled),
        "dedup_diff_threshold": float(diff_threshold),
        "dedup_resize_width": -1 if resize_width is None else int(resize_width),
        "dedup_max_skipped_run": -1 if max_skipped_run is None else int(max_skipped_run),
        "dedup_input_frame_pairs": int(input_frame_pairs),
        "dedup_kept_frame_pairs": int(len(kept_indices)),
        "dedup_removed_frame_pairs": int(duplicate_frame_pairs),
        "dedup_removed_ratio": float(duplicate_frame_pairs / input_frame_pairs) if input_frame_pairs else 0.0,
        "dedup_diff_score_mean": float(np.mean(finite_diff_scores)) if finite_diff_scores.size else np.nan,
        "dedup_diff_score_p95": float(np.percentile(finite_diff_scores, 95)) if finite_diff_scores.size else np.nan,
        "output_fps": float(output_fps),
    }
    return kept_indices, info


# 関数メモ: load_outdoor_video_names
# - outdoor_list.txt を読み、屋外動画として扱う元ファイル名の集合を作ります。
# - ファイル配置だけでは indoor/outdoor が分からない場合の環境ラベル付けに使います。

def load_outdoor_video_names(outdoor_list_path: Path = OUTDOOR_LIST_PATH) -> set[str]:
    if not outdoor_list_path.exists():
        return set()
    return {line.strip() for line in outdoor_list_path.read_text(encoding="utf-8").splitlines() if line.strip()}


# 関数メモ: load_frame_cut_config
# - frame_cut.txt から、動画ごとの切り出し開始・終了指定を読みます。
# - 手動で指定した trim 範囲を自動前処理に反映するための設定読み込みです。

def load_frame_cut_config(frame_cut_path: Path = FRAME_CUT_PATH) -> dict[str, dict[str, int]]:
    if not frame_cut_path.exists():
        return {}

    frame_cut_config: dict[str, dict[str, int]] = {}
    for line_number, raw_line in enumerate(frame_cut_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split()
        if len(parts) != 3:
            raise ValueError(f"Invalid frame_cut.txt line {line_number}: {raw_line!r}")

        video_name, front_cut_text, rear_cut_text = parts
        try:
            front_cut_frames = int(front_cut_text)
            rear_cut_frames = int(rear_cut_text)
        except ValueError as error:
            raise ValueError(f"Invalid frame_cut.txt frame count at line {line_number}: {raw_line!r}") from error

        if front_cut_frames < 0 or rear_cut_frames < 0:
            raise ValueError(f"frame_cut.txt frame counts must be non-negative at line {line_number}: {raw_line!r}")

        frame_cut_config[video_name] = {
            "front": front_cut_frames,
            "rear": rear_cut_frames,
        }
    return frame_cut_config


# 関数メモ: infer_environment
# - 動画名が outdoor_list に含まれるかで indoor/outdoor を推定します。
# - 出力先ディレクトリと manifest の environment 列を一貫させます。

def infer_environment(video_path: Path, outdoor_video_names: set[str]) -> str:
    return "outdoor" if video_path.stem in outdoor_video_names else "indoor"


# 関数メモ: find_movie_preprocess_targets
# - 元動画ディレクトリを走査し、前処理対象の動画と category/environment を一覧化します。
# - バッチ処理本体がこのリストだけを順に処理すればよい形にします。

def find_movie_preprocess_targets() -> list[dict[str, Any]]:
    targets: list[dict[str, Any]] = []
    outdoor_video_names = load_outdoor_video_names()
    for category, video_dir in INPUT_VIDEO_DIRS.items():
        if not video_dir.exists():
            continue
        for video_path in sorted(video_dir.glob("*.mp4")):
            if video_path.is_file():
                targets.append({
                    "category": category,
                    "environment": infer_environment(video_path, outdoor_video_names),
                    "video_path": video_path,
                })
    return targets


# 関数メモ: build_movie_preprocess_output_paths
# - 入力動画から front/rear/audio/manifest 用の出力パスを組み立てます。
# - category/environment/sample_id の命名を1か所に集約し、保存先の揺れを防ぎます。

def build_movie_preprocess_output_paths(
    video_path: Path,
    category: str,
    environment: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> tuple[Path, Path, Path]:
    output_dir = output_base_dir / category / environment
    front_output_path = output_dir / f"{video_path.stem}_front{video_path.suffix}"
    rear_output_path = output_dir / f"{video_path.stem}_rear{video_path.suffix}"
    audio_output_path = output_dir / f"{video_path.stem}_audio.wav"
    return front_output_path, rear_output_path, audio_output_path


# 関数メモ: find_existing_movie_preprocess_files
# - 同じ sample_id に対する既存の前処理成果物を探します。
# - 再処理や削除が必要かを判断するため、現在の出力状態を確認します。

def find_existing_movie_preprocess_files(
    video_name: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> list[Path]:
    if not output_base_dir.exists():
        return []

    existing_paths = [
        path
        for path in output_base_dir.rglob("*")
        if path.is_file()
        and path.name != MOVIE_PREPROCESS_MANIFEST_PATH.name
        and (path.stem == video_name or path.stem.startswith((f"{video_name}_", f"{video_name}-")))
    ]
    return sorted(existing_paths)


# 関数メモ: load_movie_preprocess_manifest
# - 過去の前処理 manifest を読み込みます。
# - 存在しない場合は空の表を返し、初回実行と再実行を同じ流れで扱います。

def load_movie_preprocess_manifest(
    manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH,
) -> pd.DataFrame:
    if not manifest_path.exists():
        return pd.DataFrame()
    return pd.read_csv(manifest_path)


# 関数メモ: build_previous_frame_cut_records
# - 過去 manifest から、動画ごとの frame cut 設定や出力状態を引ける辞書を作ります。
# - 設定が変わった動画だけ再処理するための比較材料です。

def build_previous_frame_cut_records(manifest_df: pd.DataFrame) -> dict[str, dict[str, Any]]:
    required_columns = {"input_video_path", "frame_cut_front_frames", "frame_cut_rear_frames"}
    if manifest_df.empty or not required_columns.issubset(manifest_df.columns):
        return {}

    records: dict[str, dict[str, Any]] = {}
    for row in manifest_df.to_dict("records"):
        if pd.isna(row.get("frame_cut_front_frames")) or pd.isna(row.get("frame_cut_rear_frames")):
            continue
        video_name = Path(str(row["input_video_path"])).stem
        record: dict[str, Any] = {
            "front": int(row["frame_cut_front_frames"]),
            "rear": int(row["frame_cut_rear_frames"]),
        }
        for column in [
            "dedup_enabled",
            "dedup_diff_threshold",
            "dedup_resize_width",
            "dedup_max_skipped_run",
        ]:
            value = row.get(column)
            if value is not None and not pd.isna(value):
                record[column] = value
        records[video_name] = record
    return records


# 関数メモ: frame_cut_record_matches
# - 今回の frame cut 設定が、過去 manifest に記録された設定と一致するかを判定します。
# - trim 指定が変わったときに古い成果物を使い回さないためのチェックです。

def frame_cut_record_matches(
    previous_record: dict[str, Any] | None,
    front_cut_frames: int,
    rear_cut_frames: int,
) -> bool:
    if previous_record is None:
        return False
    return previous_record.get("front") == front_cut_frames and previous_record.get("rear") == rear_cut_frames


# 関数メモ: manifest_bool
# - manifest 上の文字列・数値・真偽値を bool として安定的に解釈します。
# - CSV 由来の `True` / `False` / `1` / `0` の揺れを吸収します。

def manifest_bool(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes"}
    return bool(value)


# 関数メモ: dedup_record_matches
# - 重複フレーム除去に関する今回設定と過去設定が一致するかを判定します。
# - 間引き設定を変えたときに必要な再処理を漏らさないための比較です。

def dedup_record_matches(
    previous_record: dict[str, Any] | None,
    enabled: bool = MOVIE_PREPROCESS_DEDUP_FRAMES,
    diff_threshold: float = MOVIE_PREPROCESS_DEDUP_DIFF_THRESHOLD,
    resize_width: int | None = MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH,
    max_skipped_run: int | None = MOVIE_PREPROCESS_DEDUP_MAX_SKIPPED_RUN,
) -> bool:
    if not enabled:
        return True
    if previous_record is None or not manifest_bool(previous_record.get("dedup_enabled", False)):
        return False

    expected_resize_width = -1 if resize_width is None else int(resize_width)
    expected_max_skipped_run = -1 if max_skipped_run is None else int(max_skipped_run)
    try:
        return (
            np.isclose(float(previous_record.get("dedup_diff_threshold")), float(diff_threshold))
            and int(float(previous_record.get("dedup_resize_width"))) == expected_resize_width
            and int(float(previous_record.get("dedup_max_skipped_run"))) == expected_max_skipped_run
        )
    except (TypeError, ValueError):
        return False


# 関数メモ: delete_movie_preprocess_files
# - 指定された前処理成果物を削除します。
# -  stale な出力を消してから再処理することで、古いファイルの混入を防ぎます。

def delete_movie_preprocess_files(paths: list[Path]) -> None:
    for path in paths:
        if path.exists():
            path.unlink()


# 関数メモ: expected_movie_preprocess_output_paths
# - manifest の1行から、本来存在すべき前処理成果物パスを復元します。
# - 現在のディレクトリ内にある余計なファイルを判定する基準にします。

def expected_movie_preprocess_output_paths(
    targets: list[dict[str, Any]],
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> set[Path]:
    expected_paths: set[Path] = set()
    for target in targets:
        expected_paths.update(
            path.resolve()
            for path in build_movie_preprocess_output_paths(
                video_path=target["video_path"],
                category=target["category"],
                environment=target["environment"],
                output_base_dir=output_base_dir,
            )
        )
    return expected_paths


# 関数メモ: iter_movie_preprocess_artifacts
# - movie_preprocess 配下の mp4 / wav / csv など成果物ファイルを列挙します。
# - 不要ファイル掃除や状態確認を一括で行うための走査関数です。

def iter_movie_preprocess_artifacts(output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR) -> list[Path]:
    artifacts: list[Path] = []
    for category in INPUT_VIDEO_DIRS:
        category_dir = output_base_dir / category
        if not category_dir.exists():
            continue
        for path in category_dir.glob("*/*"):
            if not path.is_file():
                continue
            if path.name.endswith(("_front.mp4", "_rear.mp4", "_audio.wav")):
                artifacts.append(path)
    return sorted(artifacts)


# 関数メモ: remove_empty_movie_preprocess_dirs
# - 成果物削除後に空になったカテゴリ・環境ディレクトリを削除します。
# - 再実行を繰り返しても出力ツリーに空ディレクトリが残りにくくします。

def remove_empty_movie_preprocess_dirs(output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR) -> None:
    for directory in sorted(
        [p for p in output_base_dir.glob("*/*") if p.is_dir()],
        key=lambda p: len(p.parts),
        reverse=True,
    ):
        try:
            directory.rmdir()
        except OSError:
            pass


# 関数メモ: cleanup_stale_movie_preprocess_outputs
# - 今回の manifest から期待されない古い成果物を削除します。
# - 入力動画や設定が変わったあと、古い前処理結果を誤って使わないための掃除です。

def cleanup_stale_movie_preprocess_outputs(
    targets: list[dict[str, Any]],
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    delete_stale: bool = MOVIE_PREPROCESS_DELETE_STALE,
) -> list[Path]:
    expected_paths = expected_movie_preprocess_output_paths(targets, output_base_dir=output_base_dir)
    stale_paths = [
        path for path in iter_movie_preprocess_artifacts(output_base_dir=output_base_dir)
        if path.resolve() not in expected_paths
    ]
    if not stale_paths:
        print("stale movie_preprocess artifacts: 0")
        return []

    action = "delete" if delete_stale else "keep"
    print(f"stale movie_preprocess artifacts: {len(stale_paths)} ({action})")
    for path in stale_paths:
        print("  stale:", path)
        if delete_stale:
            path.unlink()

    if delete_stale:
        remove_empty_movie_preprocess_dirs(output_base_dir=output_base_dir)
    return stale_paths


# 関数メモ: has_audio_stream
# - ffprobe で動画に音声ストリームが含まれるかを確認します。
# - 音声がない動画に対して ffmpeg 抽出を実行して失敗扱いにするのを避けます。

def has_audio_stream(video_path: Path) -> bool:
    command = [
        FFPROBE_PATH,
        "-v",
        "error",
        "-select_streams",
        "a",
        "-show_entries",
        "stream=index",
        "-of",
        "csv=p=0",
        str(video_path),
    ]
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    return bool(completed.stdout.strip())


# 関数メモ: extract_trimmed_audio_to_wav
# - ffmpeg を使い、動画の指定区間からモノラル wav 音声を抽出します。
# - front/rear 動画と同じ trim 範囲に合わせ、音声特徴の時刻を動画とそろえます。

def extract_trimmed_audio_to_wav(
    video_path: Path,
    output_wav_path: Path,
    start_sec: float,
    duration_sec: float,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
    sample_rate: int = AUDIO_SAMPLE_RATE,
    channels: int = AUDIO_MONO_CHANNELS,
) -> str:
    if duration_sec <= 0:
        return "skipped_zero_duration"
    if not overwrite and output_wav_path.exists():
        return "skipped_existing_audio"
    if not has_audio_stream(video_path):
        return "skipped_missing_audio_stream"

    ensure_parent_directory(output_wav_path)
    command = [
        FFMPEG_PATH,
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-ss",
        f"{max(start_sec, 0.0):.6f}",
        "-i",
        str(video_path),
        "-t",
        f"{max(duration_sec, 0.0):.6f}",
        "-vn",
        "-ac",
        str(channels),
        "-ar",
        str(sample_rate),
        "-c:a",
        "pcm_s16le",
        str(output_wav_path),
    ]
    subprocess.run(command, check=True)
    return "written"


## 2. 前処理実行


In [ ]:
# ------------------------------------------------------------
# セル概要: 前処理本体とバッチ実行
# ------------------------------------------------------------
# - 1本の元動画を前後カメラ動画と音声wavへ変換し、その処理をデータセット全体へ適用します。
# - 前回 manifest と設定を比較し、再処理が必要なものだけを処理できるようにしています。
# ------------------------------------------------------------

# 関数メモ: preprocess_movie_to_front_rear_audio
# - 1本の元動画を読み、trim・黒帯除去・front/rear分割・音声抽出・manifest行作成まで行います。
# - この関数が動画前処理の中心で、成功・skip・error の状態もここで決まります。

def preprocess_movie_to_front_rear_audio(
    video_path: Path,
    category: str,
    environment: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
    frame_cut_config: dict[str, dict[str, int]] | None = None,
    previous_frame_cut_records: dict[str, dict[str, Any]] | None = None,
) -> dict[str, Any]:
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 30.0
    front_output_path, rear_output_path, audio_output_path = build_movie_preprocess_output_paths(
        video_path=video_path,
        category=category,
        environment=environment,
        output_base_dir=output_base_dir,
    )

    result: dict[str, Any] = {
        "category": category,
        "environment": environment,
        "input_video_path": str(video_path),
        "front_output_path": str(front_output_path),
        "rear_output_path": str(rear_output_path),
        "audio_output_path": str(audio_output_path),
        "input_frame_count": int(metadata["frame_count"]),
        "input_fps": float(metadata["fps"]),
        "input_width": int(metadata["width"]),
        "input_height": int(metadata["height"]),
        "input_duration_sec": float(metadata["duration_sec"]),
        "dedup_enabled": bool(MOVIE_PREPROCESS_DEDUP_FRAMES),
        "dedup_diff_threshold": float(MOVIE_PREPROCESS_DEDUP_DIFF_THRESHOLD),
        "dedup_resize_width": -1 if MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH is None else int(MOVIE_PREPROCESS_DEDUP_RESIZE_WIDTH),
        "dedup_max_skipped_run": -1 if MOVIE_PREPROCESS_DEDUP_MAX_SKIPPED_RUN is None else int(MOVIE_PREPROCESS_DEDUP_MAX_SKIPPED_RUN),
    }

    frame_cut_has_explicit_config = video_path.stem in (frame_cut_config or {})
    frame_cut = (frame_cut_config or {}).get(video_path.stem, {})
    front_cut_frames = int(frame_cut.get("front", 0))
    rear_cut_frames = int(frame_cut.get("rear", 0))
    frame_cut_start_index = front_cut_frames
    frame_cut_end_index = metadata["frame_count"] - rear_cut_frames
    remaining_after_initial_frame_cut = max(frame_cut_end_index - frame_cut_start_index, 0)
    result.update({
        "frame_cut_front_frames": front_cut_frames,
        "frame_cut_rear_frames": rear_cut_frames,
        "frame_cut_start_frame_index": int(frame_cut_start_index),
        "frame_cut_end_frame_index": int(max(frame_cut_start_index, frame_cut_end_index)),
        "remaining_frame_pairs_after_frame_cut": int(remaining_after_initial_frame_cut),
    })

    existing_preprocess_files = find_existing_movie_preprocess_files(
        video_path.stem,
        output_base_dir=output_base_dir,
    )
    previous_frame_cut_record = (previous_frame_cut_records or {}).get(video_path.stem)
    if previous_frame_cut_record is not None:
        result.update({
            "previous_frame_cut_front_frames": int(previous_frame_cut_record["front"]),
            "previous_frame_cut_rear_frames": int(previous_frame_cut_record["rear"]),
        })
        if "dedup_enabled" in previous_frame_cut_record:
            result.update({
                "previous_dedup_enabled": manifest_bool(previous_frame_cut_record.get("dedup_enabled")),
                "previous_dedup_diff_threshold": previous_frame_cut_record.get("dedup_diff_threshold"),
                "previous_dedup_resize_width": previous_frame_cut_record.get("dedup_resize_width"),
                "previous_dedup_max_skipped_run": previous_frame_cut_record.get("dedup_max_skipped_run"),
            })

    if not overwrite and existing_preprocess_files:
        result["existing_preprocess_files"] = ";".join(str(path) for path in existing_preprocess_files)
        frame_cut_matches = frame_cut_record_matches(previous_frame_cut_record, front_cut_frames, rear_cut_frames)
        dedup_matches = dedup_record_matches(previous_frame_cut_record)
        if frame_cut_matches and dedup_matches:
            result.update({
                "status": "skipped_existing_preprocess_file",
                "audio_status": "not_started",
                "frame_cut_record_status": "matched",
                "dedup_record_status": "matched" if MOVIE_PREPROCESS_DEDUP_FRAMES else "not_required",
            })
            return result
        if previous_frame_cut_record is None and not frame_cut_has_explicit_config and not MOVIE_PREPROCESS_DEDUP_FRAMES:
            result.update({
                "status": "skipped_existing_preprocess_file",
                "audio_status": "not_started",
                "frame_cut_record_status": "assumed_default",
                "dedup_record_status": "not_required",
            })
            return result

        delete_movie_preprocess_files(existing_preprocess_files)
        result.update({
            "frame_cut_record_status": "matched" if frame_cut_matches else "missing_or_changed",
            "dedup_record_status": "matched" if dedup_matches else "missing_or_changed",
            "reprocess_reason": "frame_cut_or_dedup_changed_or_unrecorded",
            "deleted_existing_preprocess_files": ";".join(str(path) for path in existing_preprocess_files),
        })

    if remaining_after_initial_frame_cut <= 0:
        result.update({
            "status": "skipped_no_frames_after_frame_cut",
            "written_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    sampled_frames = iter_sampled_frames(
        video_path,
        start_frame_index=frame_cut_start_index,
        end_frame_index=frame_cut_end_index,
    )
    result["sampled_frame_count"] = int(len(sampled_frames))
    if not sampled_frames:
        result.update({
            "status": "skipped_no_sampled_frames",
            "audio_status": "not_started",
        })
        return result

    _, crop_info = preprocess_stacked_frames_remove_letterbox(sampled_frames)
    trim_info = estimate_full_video_uniform_trim(
        video_path,
        crop_info,
        start_frame_index=frame_cut_start_index,
        end_frame_index=frame_cut_end_index,
    )
    result.update({
        "crop_left": int(crop_info["crop_left"]),
        "crop_right": int(crop_info["crop_right"]),
        "cropped_width": int(crop_info["cropped_width"]),
        "removed_left_px": int(crop_info["removed_left_px"]),
        "removed_right_px": int(crop_info["removed_right_px"]),
    })
    result.update(trim_info)

    start_index = frame_cut_start_index + trim_info["removed_from_start"]
    end_index = frame_cut_end_index - trim_info["removed_from_end"]
    remaining_after_trim = max(end_index - start_index, 0)
    result.update({
        "trim_start_frame_index": int(start_index),
        "trim_end_frame_index": int(max(start_index, end_index)),
        "remaining_frame_pairs_after_trim": int(remaining_after_trim),
    })

    if remaining_after_trim <= 0:
        result.update({
            "status": "skipped_no_frames_after_trim",
            "written_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    kept_frame_indices, dedup_info = select_nonduplicate_frame_indices(
        video_path=video_path,
        start_frame_index=start_index,
        end_frame_index=end_index,
        crop_info=crop_info,
        native_fps=native_fps,
    )
    kept_frame_index_set = set(kept_frame_indices)
    output_fps = float(dedup_info["output_fps"])
    source_trim_duration_sec = float(remaining_after_trim / native_fps) if native_fps > 0 else 0.0
    result.update(dedup_info)
    result["source_trim_duration_sec"] = source_trim_duration_sec

    if not kept_frame_indices:
        result.update({
            "status": "skipped_no_frames_after_dedup",
            "written_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    front_writer = None
    rear_writer = None
    frame_pair_index = 0
    written_frame_pairs = 0

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break
            if frame_pair_index > kept_frame_indices[-1]:
                break

            should_write = frame_pair_index in kept_frame_index_set
            if should_write:
                cropped_frame_bgr = apply_side_crop(
                    stacked_frame_bgr,
                    crop_info["crop_left"],
                    crop_info["crop_right"],
                )
                front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)

                if front_writer is None or rear_writer is None:
                    front_writer = create_mp4_writer(front_output_path, output_fps, front_frame_bgr.shape)
                    rear_writer = create_mp4_writer(rear_output_path, output_fps, rear_frame_bgr.shape)
                    result.update({
                        "output_width": int(front_frame_bgr.shape[1]),
                        "output_height": int(front_frame_bgr.shape[0]),
                    })

                front_writer.write(front_frame_bgr)
                rear_writer.write(rear_frame_bgr)
                written_frame_pairs += 1

            frame_pair_index += 1
    finally:
        capture.release()
        if front_writer is not None:
            front_writer.release()
        if rear_writer is not None:
            rear_writer.release()

    trim_start_sec = float(start_index / native_fps)
    trim_duration_sec = source_trim_duration_sec
    audio_status = extract_trimmed_audio_to_wav(
        video_path=video_path,
        output_wav_path=audio_output_path,
        start_sec=trim_start_sec,
        duration_sec=trim_duration_sec,
        overwrite=overwrite,
    )

    result.update({
        "status": "written",
        "written_frame_pairs": int(written_frame_pairs),
        "trim_start_sec": trim_start_sec,
        "trim_duration_sec": trim_duration_sec,
        "audio_status": audio_status,
    })
    return result


# 関数メモ: run_movie_preprocess_for_dataset
# - 対象動画リスト全体に前処理を適用し、manifest を保存します。
# - 上書き設定や stale 削除を含め、データセット単位の再現可能な前処理を実行します。

def run_movie_preprocess_for_dataset(
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
) -> pd.DataFrame:
    targets = find_movie_preprocess_targets()
    frame_cut_config = load_frame_cut_config()
    previous_manifest_df = load_movie_preprocess_manifest(manifest_path)
    previous_frame_cut_records = build_previous_frame_cut_records(previous_manifest_df)
    print("movie preprocess target count:", len(targets))
    print("frame cut target count:", len(frame_cut_config))
    print("previous frame cut record count:", len(previous_frame_cut_records))
    cleanup_stale_movie_preprocess_outputs(
        targets=targets,
        output_base_dir=output_base_dir,
        delete_stale=MOVIE_PREPROCESS_DELETE_STALE,
    )
    rows: list[dict[str, Any]] = []

    for target_index, target in enumerate(targets, start=1):
        category = target["category"]
        environment = target["environment"]
        video_path = target["video_path"]
        print(f"[{target_index}/{len(targets)}] {category}/{environment}: {video_path.name}")
        try:
            result = preprocess_movie_to_front_rear_audio(
                video_path=video_path,
                category=category,
                environment=environment,
                output_base_dir=output_base_dir,
                overwrite=overwrite,
                frame_cut_config=frame_cut_config,
                previous_frame_cut_records=previous_frame_cut_records,
            )
        except Exception as error:
            front_output_path, rear_output_path, audio_output_path = build_movie_preprocess_output_paths(
                video_path=video_path,
                category=category,
                environment=environment,
                output_base_dir=output_base_dir,
            )
            result = {
                "category": category,
                "environment": environment,
                "input_video_path": str(video_path),
                "front_output_path": str(front_output_path),
                "rear_output_path": str(rear_output_path),
                "audio_output_path": str(audio_output_path),
                "status": "error",
                "audio_status": "error",
                "error": repr(error),
            }

        rows.append(result)
        print(
            "  status:",
            result.get("status"),
            "written_frame_pairs:",
            result.get("written_frame_pairs"),
            "dedup_removed_frame_pairs:",
            result.get("dedup_removed_frame_pairs"),
            "output_fps:",
            result.get("output_fps"),
            "audio_status:",
            result.get("audio_status"),
        )

    manifest_df = pd.DataFrame(rows)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_df.to_csv(manifest_path, index=False)
    print("movie preprocess manifest saved to:", manifest_path)
    if not manifest_df.empty and "status" in manifest_df:
        print("status counts:")
        print(manifest_df["status"].value_counts(dropna=False).to_dict())
    if not manifest_df.empty and "audio_status" in manifest_df:
        print("audio status counts:")
        print(manifest_df["audio_status"].value_counts(dropna=False).to_dict())
    return manifest_df


if VIDEO_PROCESSING_ENABLED:
    movie_preprocess_manifest_df = run_movie_preprocess_for_dataset()
    display(movie_preprocess_manifest_df.head())
else:
    print("Skip movie preprocessing because VIDEO_PROCESSING_ENABLED is False.")
